<a href="https://colab.research.google.com/github/Saishiva-hub/News-Ninja-Agent/blob/main/NewsNinja21aug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain-groq langgraph langchain-community tavily-python

In [ ]:
# SET UP THE API KEY--------------------------------

In [ ]:
import os
import getpass

print("Enter your Groq API Key: ")
os.environ['GROQ_API_KEY'] = getpass.getpass()

print('Enter your Tavily API Key: ')
os.environ['TAVILY_API_KEY'] = getpass.getpass()


Enter your Groq API Key: 
··········
Enter your Tavily API Key: 
··········


In [ ]:
# DEFINE THE TOOLS
# --------------------------------------------------------------------------------------------------------------

In [ ]:
import warnings
warnings.filterwarnings('ignore',category=DeprecationWarning)
# DeprecationWarning -> "This feature may be removed in the future"

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

@tool
def search_news(query: str) -> str:
  """Search the web for the latest news articles on a given topic, simply."""
  search = TavilySearchResults(
      max_result = 5,
      search_depth = "advanced",
      include_answer = True
  )
  results = search.invoke({'query':query})

  if not results:
    return " NO RESULTS FOUND FOR THE GIVEN QUERY"

  formatted = []
  for i , result in enumerate(results,1):
    title = result.get("title","No Title")
    url = result.get("url","")
    content = result.get("content","No content")
    formatted.append(
        f"**Article {i}:** {title}\n"
        f" URL: {url}\n"
        f" Summary: {content}\n"
    )

    return "\n".join(formatted)

In [ ]:
from typing import TypedDict
from datetime import datetime
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , END


# Define State
class AgentState(TypedDict):
  topic: str
  search_query : str
  search_results: str
  digest: str


# Define the nodes -----------------------------
# Node 1 - Parser Node
def parse_topic(state: AgentState)-> dict:
  """Parse & refine user topic into an optimized search query."""
  llm = ChatGroq(model = "openai/gpt-oss-120b",
                 temperature = 0)
  prompt = (
      " You are a search query optimizer. Give the following topic, create a concise, "
      " effective search query to find the latest news about it."
      " Return Only the optimized search query, nothing else.\n\n"
      f"Topic: {state['topic']}"
  )
  response = llm.invoke(prompt)
  search_query = response.content.strip()
  print(f"\n[Node: parse_topic] optimised seach query: {search_query}")
  return {"search_query": search_query}


# Node2- Searcher Node
def search_web(state: AgentState)-> dict:
  """Use Tavily to search for the news article"""
  query = state.get("search_query",state['topic'])
  print(f"\n[Node: search_web] Searching the web for: {query}")
  results = search_news.invoke({"query": query})
  print(f"Found {results.count('Article')} articles")
  return {"search_results": results}

# Node3- Writer
def write_digest(state: AgentState)-> dict:
  """ Use Groq LLM to syntesize a polished news digest."""
  llm = ChatGroq(model= "openai/gpt-oss-120b",
                 temperature= 0)
  prompt = (
      "You are NewsNinja, an elite AI news analyst. Based on the following search "
      "results. create a comprehensive yet concise news digest. \n\n"
      "FORMAT YOUR RESPONSE AS: \n"
      "** NewsNinja Digest **\n\n"
      f"**Topic: ** {state['topic']}\n"
      f"**Date: ** {datetime.now().strftime('%B %d, %Y')}\n\n"
      "---------------------------------------\n\n"
      "Then provide: \n\n"
      "1. A brief **Executive Summary** (2-3 sentences)\n"
      "2. **Key Headlines** with bullet points\n"
      "3. **Analysis & Insights** (what this means)\n"
      "4. **Sources** (list the URLs)\n\n"
      "----------------------------------------\n\n"
      f"SEARCH RESULTS: \n{state['search_results']}"
  )
  response = llm.invoke(prompt)
  print("\n[Node: write_digest] Digest written!!!")
  return {'digest': response.content}

# Compile GRAPH

def build_graph() -> StateGraph:
  graph = StateGraph(AgentState)
  graph.add_node("parser",parse_topic)
  graph.add_node("searcher",search_web)
  graph.add_node("writer",write_digest)

  graph.set_entry_point("parser")
  graph.add_edge("parser","searcher")
  graph.add_edge("searcher","writer")
  graph.add_edge("writer",END)

  return graph.compile()

# Build it

app = build_graph()


In [ ]:
topic = input('Enter a news topic to research: ')

if topic.strip():
  print("="*60)
  print(f"Starting pipeline for: {topic}")
  print("-"*60)

  # INVOKE THE GRAPH
  result = app.invoke({"topic": topic})

  print("\n" + "="*60)
  print(result['digest'])
  print("="*60)

else:
  print("No topic is provided. Please try again")

Enter a news topic to research: cricket
Starting pipeline for: cricket
------------------------------------------------------------

[Node: parse_topic] optimised seach query: cricket latest news

[Node: search_web] Searching the web for: cricket latest news


/tmp/ipykernel_1203/1016148352.py:11: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(


Found 1 articles

[Node: write_digest] Digest written!!!

** NewsNinja Digest **  

**Topic:** cricket  
**Date:** August 25, 2026  

---  

### Executive Summary  
India seized control of the 2nd Test in Colombo as wicket‑keeper‑batter Dhruv Jurel smashed a maiden Test century, becoming only the third Indian batsman at No. 7 or lower to reach three figures. Off the field, the cricketing world is buzzing with coaching wars ahead of the Ashes, disciplinary actions from the ICC, and political drama surrounding Pakistan’s Imran Khan.  

---  

### Key Headlines  
- **Dhruv Jurel’s Test century propels India to a dominant position in the 2nd Test vs Sri Lanka (Colombo, Day 2).**  
- **Australia and England vie to hire former CSK head coach (likely Stephen Fleming) ahead of the upcoming Ashes series.**  
- **ICC hands punishment to Yashasvi Jaiswal for head‑butting a Sri Lankan bowler; Jaiswal publicly admits his mistake.**  
- **Afghanistan include all‑rounder Naveen‑Ul‑Haq in the squad fo